# Phase 1 — Data Exploration & Quality Audit (Kaggle) 
**Project:** Explainable-NIDS — GraphIDS reproduction on NF-UNSW-NB15-v3  
**Platform:** Kaggle CPU session (pandas only — no GPU needed)  
**Input:** private Kaggle dataset nf-unsw-nb15-v3 (re-attach via Add Input)  
**Purpose:** produce the data quality report that Phase 2 will cite.  
**Gate:** this notebook must run end-to-end and save data_quality_report.md before Phase 2 starts.

In [ ]:
# === CELL 2: STAGE DATASET (same as Phase 0 — /kaggle/input is read-only) ===
import glob, os, shutil

src_csv = glob.glob('/kaggle/input/**/NF-UNSW-NB15-v3.csv', recursive=True)
src_feat = glob.glob('/kaggle/input/**/NetFlow_v3_Features.csv', recursive=True)
print("Found dataset CSV :", src_csv)
print("Found features CSV:", src_feat)
assert src_csv and src_feat, "Dataset not attached! Check Input panel -> Add Input, then re-run."

os.makedirs('data/NF-UNSW-NB15-v3', exist_ok=True)
for f in src_csv + src_feat:
    dst = os.path.join('data/NF-UNSW-NB15-v3', os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy(f, dst)
        print("Copied:", dst)

!ls -lh data/NF-UNSW-NB15-v3/

In [ ]:
# === CELL 3: FULL LOAD — all rows this time, not just 1000 ===
import pandas as pd
import numpy as np
import time

t0 = time.time()
df = pd.read_csv('data/NF-UNSW-NB15-v3/NF-UNSW-NB15-v3.csv', low_memory=False)
load_time = time.time() - t0

print(f"Loaded in {load_time:.1f}s")
print("Shape:", df.shape)

mem_gb = df.memory_usage(deep=True).sum() / 1e9
print(f"Memory usage: {mem_gb:.2f} GB")

findings = {}
findings['shape'] = list(df.shape)
findings['memory_gb'] = round(mem_gb, 2)
findings['load_seconds'] = round(load_time, 1)

In [ ]:
# === CELL 4: DTYPES & COLUMN GROUPS ===
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
object_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric columns ({len(numeric_cols)}):")
print(numeric_cols)
print(f"\nObject columns ({len(object_cols)}):")
print(object_cols)

# Identifier columns -> these become GRAPH NODES in Phase 2 (GraphIDS)
id_cols = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'L4_SRC_PORT', 'L4_DST_PORT', 'PROTOCOL']
print("\nIdentifier columns (future graph nodes):")
print(df[id_cols].dtypes)

findings['numeric_cols'] = len(numeric_cols)
findings['object_cols'] = object_cols

In [ ]:
# === CELL 5: TARGET AUDIT — Label vs Attack (FIXED: Label is int 0/1, not string) ===
print("=== Label (binary) ===")
print(df['Label'].value_counts())

print("\n=== Attack (multi-class) ===")
print(df['Attack'].value_counts())

print("\n=== Crosstab: Label x Attack ===")
ct = pd.crosstab(df['Label'], df['Attack'])
print(ct)

# Consistency check — Label is NUMERIC (0=benign, 1=attack), Attack is string
mismatch_a = df[(df['Label'] == 0) & (df['Attack'] != 'Benign')]
mismatch_b = df[(df['Label'] == 1) & (df['Attack'] == 'Benign')]
print(f"\nLabel=0 but Attack!=Benign : {len(mismatch_a):,} rows")
print(f"Label=1 but Attack=Benign  : {len(mismatch_b):,} rows")

findings['label_counts'] = df['Label'].value_counts().to_dict()
findings['attack_counts'] = df['Attack'].value_counts().to_dict()
findings['label_attack_mismatches'] = int(len(mismatch_a) + len(mismatch_b))
findings['label_dtype'] = str(df['Label'].dtype)

In [ ]:
# === CELL 6: MISSING & INFINITE VALUES ===
nan_counts = df.isna().sum()
nan_offenders = nan_counts[nan_counts > 0].sort_values(ascending=False)
print("=== Columns with NaN ===")
if len(nan_offenders) == 0:
    print("None — no missing values.")
else:
    print(nan_offenders)

num_df = df.select_dtypes(include=[np.number])
inf_counts = np.isinf(num_df).sum()
inf_offenders = inf_counts[inf_counts > 0]
print("\n=== Columns with +/- inf ===")
if len(inf_offenders) == 0:
    print("None.")
else:
    print(inf_offenders)

findings['nan_offenders'] = nan_offenders.to_dict()
findings['inf_offenders'] = inf_offenders.to_dict()

In [ ]:
# === CELL 7: CONSTANT & NEAR-CONSTANT COLUMNS ===
nunique = df.nunique(dropna=False)
constant_cols = nunique[nunique <= 1].index.tolist()
print("=== Constant columns (1 unique value) ===")
print(constant_cols if constant_cols else "None.")

near_constant = {}
for col in numeric_cols:
    vc = df[col].value_counts(normalize=True, dropna=False)
    if vc.iloc[0] > 0.999 and col not in constant_cols:
        near_constant[col] = round(vc.iloc[0] * 100, 2)

print("\n=== Near-constant numeric columns (>99.9% one value) ===")
for col, pct in sorted(near_constant.items(), key=lambda x: -x[1]):
    print(f"{col:35s} {pct:6.2f}% one value")

findings['constant_cols'] = constant_cols
findings['near_constant_cols'] = near_constant

In [ ]:
# === CELL 8: DUPLICATES ===
dup_full = df.duplicated().sum()
print(f"Full-row duplicates: {dup_full:,}")

key_cols = ['FLOW_START_MILLISECONDS', 'IPV4_SRC_ADDR', 'IPV4_DST_ADDR',
            'L4_SRC_PORT', 'L4_DST_PORT', 'PROTOCOL']
dup_key = df.duplicated(subset=key_cols).sum()
print(f"Flow-key duplicates ({', '.join(key_cols)}): {dup_key:,}")

findings['duplicates_full_row'] = int(dup_full)
findings['duplicates_flow_key'] = int(dup_key)

In [ ]:
# === CELL 9: TEMPORAL COVERAGE ===
ts = pd.to_datetime(df['FLOW_START_MILLISECONDS'], unit='ms')
print("First flow :", ts.min())
print("Last flow  :", ts.max())
print(f"Span       : {ts.max() - ts.min()}")

per_day = ts.dt.floor('D').value_counts().sort_index()
print("\n=== Flows per day ===")
print(per_day)

# Is the file time-sorted? (matters for temporal splits in Phase 2)
sorted_check = df['FLOW_START_MILLISECONDS'].is_monotonic_increasing
print(f"\nFile time-sorted: {sorted_check}")

findings['time_min'] = str(ts.min())
findings['time_max'] = str(ts.max())
findings['time_sorted'] = bool(sorted_check)

In [ ]:
# === CELL 10: DEGENERATE & SPARSE FLOWS ===
zero_pkts = (df['IN_PKTS'] + df['OUT_PKTS'] == 0).sum()
zero_dur = (df['FLOW_DURATION_MILLISECONDS'] == 0).sum()
print(f"Flows with 0 total packets : {zero_pkts:,} ({zero_pkts/len(df)*100:.2f}%)")
print(f"Flows with 0 duration      : {zero_dur:,} ({zero_dur/len(df)*100:.2f}%)")

sparse_cols = ['ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE',
               'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE']
print("\n=== Sparse protocol columns (% nonzero) ===")
for col in sparse_cols:
    pct = (df[col] != 0).mean() * 100
    print(f"{col:20s} {pct:6.3f}% nonzero")

findings['zero_packet_flows'] = int(zero_pkts)
findings['zero_duration_flows'] = int(zero_dur)
findings['sparse_cols_nonzero_pct'] = {c: round((df[c] != 0).mean()*100, 3) for c in sparse_cols}

In [ ]:
# === CELL 11: GRAPH PREVIEW — how big will the Phase 2 graph be? ===
n_src = df['IPV4_SRC_ADDR'].nunique()
n_dst = df['IPV4_DST_ADDR'].nunique()
n_pairs = df.groupby(['IPV4_SRC_ADDR', 'IPV4_DST_ADDR']).ngroups
avg_flows_per_pair = len(df) / n_pairs

print(f"Unique source IPs      : {n_src:,}")
print(f"Unique destination IPs : {n_dst:,}")
print(f"Unique (src,dst) pairs : {n_pairs:,}")
print(f"Avg flows per pair     : {avg_flows_per_pair:,.1f}")

findings['graph'] = {
    'unique_src_ips': int(n_src),
    'unique_dst_ips': int(n_dst),
    'unique_pairs': int(n_pairs),
    'avg_flows_per_pair': round(avg_flows_per_pair, 1),
}

In [ ]:
# === CELL 12: LLM NARRATION — Groq summarizes the audit ===
import json
!pip install -q groq
from kaggle_secrets import UserSecretsClient
from groq import Groq
from IPython.display import Markdown, display

client = Groq(api_key=UserSecretsClient().get_secret("GROQ_API_KEY"))

prompt = f"""You are a senior data scientist. Below is a JSON summary of a data
quality audit of the NF-UNSW-NB15-v3 NetFlow intrusion detection dataset.

{json.dumps(findings, indent=2, default=str)}

Write a plain-language summary for a project report:
1. Describe the dataset size and composition.
2. Flag anything that threatens model training (missing data, duplicates,
   label inconsistencies, degenerate flows).
3. End with a one-line GO / NO-GO recommendation for the training phase.
Be concise. Do not invent numbers that are not in the JSON."""

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": "You are a senior data scientist writing a dataset audit report. Be concise and factual."},
        {"role": "user", "content": prompt}
    ],
    temperature=0.1,
)
llm_summary = response.choices[0].message.content
display(Markdown(llm_summary))

In [ ]:
# === CELL 13: SAVE ARTIFACT (fixed — no nested code fences) ===
report = "# Data Quality Report — NF-UNSW-NB15-v3\n"
report += "_Generated by Phase1_Data_Exploration.ipynb on Kaggle_\n\n"
report += "## Raw findings\n"
report += json.dumps(findings, indent=2, default=str)
report += "\n\n## LLM summary\n"
report += llm_summary + "\n"
report += "\n## Remediation plan (executed in Phase 2 preprocessing)\n"
report += "- inf/NaN in *_SECOND_BYTES: caused by zero-duration flows (bytes/duration division). Recompute with duration >= 1ms or drop columns.\n"
report += "- Duplicates: dedup decision to be documented before training.\n"
report += "- Zero-duration flows: 5.18% of data; check attack class balance before dropping.\n"
report += "- Time order: sort by FLOW_START_MILLISECONDS before temporal split.\n"
report += "- Label consistency: VERIFIED (0 mismatches after dtype correction).\n"

with open('/kaggle/working/data_quality_report.md', 'w') as f:
    f.write(report)
print("data_quality_report.md saved ✔")

# Phase 1 Gate — every box ✔ before Phase 2  
[ ✔ ] Cell 2: dataset staged (551 MB visible)  
[ ✔ ] Cell 3: full dataset loaded — real shape printed  
[ ✔ ] Cell 5: Label × Attack crosstab produced and interpreted  
[ ✔ ] Cell 6: NaN/inf audit done, offenders recorded (or clean bill)  
[ ✔ ] Cell 7: constant/near-constant columns quantified  
[ ✔ ] Cell 8: duplicates quantified  
[ ✔ ] Cell 9: temporal coverage + sortedness checked  
[ ✔ ] Cell 10: degenerate & sparse flows quantified  
[ ✔ ] Cell 11: graph-size preview computed  
[ ✔ ] Cell 12: LLM narration generated  
[ ✔ ] Cell 13: data_quality_report.md saved  
[ ] Save & Run All → push to GitHub as notebooks/Phase1_Data_Exploration.ipynb (WITH the .ipynb extension!)  
[ ] README updated with Phase 1 status + Kaggle link  